In [2]:
import sys 
print(sys.executable)

/Users/yashralhan/Projects/bearing-fault-detection/venv/bin/python


In [3]:
import numpy as np
data = np.load('../data/processed/mafaulda_windows.npz', allow_pickle=True)
X, y, loc_tags = data['X'], data['y'], data['loc_tags']
train_idx, test_idx = data['train_idx'].tolist(), data['test_idx'].tolist()

In [4]:
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

In [5]:
X_train_raw = X[train_idx].astype(np.float32)
X_test_raw = X[test_idx].astype(np.float32)

le = LabelEncoder()
y_train_raw = le.fit_transform(y[train_idx])
y_test_raw = le.transform(y[test_idx])

train_std = X_train_raw.std()
X_train_raw /= train_std
X_test_raw /= train_std

print(X_train_raw.shape, X_test_raw.shape, le.classes_)

(73447, 4096) (68002, 4096) ['ball_fault' 'cage_fault' 'normal' 'outer_race']


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).unsqueeze(1)   # (N, 1, 4096) - 1 input channel
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = WindowDataset(X_train_raw, y_train_raw)
test_ds = WindowDataset(X_test_raw, y_test_raw)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)


In [8]:
class FaultCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=64, stride=2), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, kernel_size=32, stride=2), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(32, 64, kernel_size=16, stride=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.squeeze(-1)
        return self.classifier(x)

model = FaultCNN(n_classes=len(le.classes_))


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 10
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
    print(f"epoch {epoch+1}/{n_epochs}  train loss: {total_loss/len(train_ds):.4f}")


epoch 1/10  train loss: 0.4529
epoch 2/10  train loss: 0.1408
epoch 3/10  train loss: 0.0985
epoch 4/10  train loss: 0.0755
epoch 5/10  train loss: 0.0581
epoch 6/10  train loss: 0.0651
epoch 7/10  train loss: 0.0416
epoch 8/10  train loss: 0.0333
epoch 9/10  train loss: 0.0269
epoch 10/10  train loss: 0.0244


In [12]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out = model(xb)
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

print("CNN trained underhang, tested overhang accuracy:", (all_preds == all_true).mean())


CNN trained underhang, tested overhang accuracy: 0.2982118172994912


In [13]:
from sklearn.metrics import classification_report
print(classification_report(le.inverse_transform(all_true), le.inverse_transform(all_preds)))

              precision    recall  f1-score   support

  ball_fault       1.00      0.12      0.21     16577
  cage_fault       0.42      0.81      0.56     22748
      normal       0.00      0.00      0.00      5929
  outer_race       0.00      0.00      0.00     22748

    accuracy                           0.30     68002
   macro avg       0.36      0.23      0.19     68002
weighted avg       0.39      0.30      0.24     68002



In [14]:
import pandas as pd
from sklearn.metrics import confusion_matrix

labels_order = le.classes_
cm_cnn = confusion_matrix(le.inverse_transform(all_true), le.inverse_transform(all_preds), labels=labels_order)
pd.DataFrame(cm_cnn, index=labels_order, columns=labels_order)


,ball_fault,cage_fault,normal,outer_race
ball_fault,1907,339,14331,0
cage_fault,0,18372,4341,35
normal,0,5929,0,0
outer_race,0,18780,3968,0


In [15]:
results_d = {
    "log+CORAL(hand features)": 0.5193,
    "raw-signal CNN": (all_preds == all_true).mean()
}
pd.Series(results_d).sort_values(ascending = False)

log+CORAL(hand features)    0.519300
raw-signal CNN              0.298212
dtype: float64

In [29]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)


In [51]:
class DANN(nn.Module):
    def __init__(self, n_classes, feat_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=64, stride=2), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, kernel_size=32, stride=2), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(32, feat_dim, kernel_size=16, stride=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.label_classifier = nn.Linear(feat_dim, n_classes)
        self.domain_classifier = nn.Sequential(
            nn.Linear(feat_dim, 32), nn.ReLU(), nn.Linear(32, 2)
        )

    def forward(self, x, lambd=1.0):
        feat = self.features(x).squeeze(-1)
        label_out = self.label_classifier(feat)
        domain_out = self.domain_classifier(grad_reverse(feat, lambd))
        return label_out, domain_out

model = DANN(n_classes=len(le.classes_), feat_dim=128).to(device)


In [38]:
domain_train = np.zeros(len(X_train_raw), dtype=np.int64)  # underhang = 0
domain_test = np.ones(len(X_test_raw), dtype=np.int64)      # overhang = 1

class DomainDataset(Dataset):
    def __init__(self, X, y_label, y_domain):
        self.X = torch.tensor(X).unsqueeze(1)
        self.y_label = torch.tensor(y_label, dtype=torch.long)
        self.y_domain = torch.tensor(y_domain, dtype=torch.long)

    def __len__(self):
        return len(self.y_domain)

    def __getitem__(self, idx):
        return self.X[idx], self.y_label[idx], self.y_domain[idx]

source_ds = DomainDataset(X_train_raw, y_train_raw, domain_train)
target_ds = DomainDataset(X_test_raw, np.zeros(len(X_test_raw), dtype=np.int64), domain_test)

source_loader = DataLoader(source_ds, batch_size=128, shuffle=True)
target_loader = DataLoader(target_ds, batch_size=128, shuffle=True)


In [52]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
label_criterion = nn.CrossEntropyLoss()
domain_criterion = nn.CrossEntropyLoss()

n_epochs = 15
warmup_epochs = 3
domain_weight = 0.5
n_batches = min(len(source_loader), len(target_loader))

for epoch in range(n_epochs):
    model.train()
    source_iter = iter(source_loader)
    target_iter = iter(target_loader)
    total_label_loss, total_domain_loss = 0, 0

    for i in range(n_batches):
        if epoch < warmup_epochs:
            lambd = 0.0
        else:
            p = (epoch - warmup_epochs) / (n_epochs - warmup_epochs)
            lambd = 2. / (1. + np.exp(-5 * p)) - 1

        xs, ys, ds = next(source_iter)
        xt, _, dt = next(target_iter)
        xs, ys, ds = xs.to(device), ys.to(device), ds.to(device)
        xt, dt = xt.to(device), dt.to(device)

        optimizer.zero_grad()
        label_out_s, domain_out_s = model(xs, lambd)
        _, domain_out_t = model(xt, lambd)

        loss_label = label_criterion(label_out_s, ys)

        if epoch < warmup_epochs:
            loss = loss_label
            loss_domain_val = 0.0
        else:
            loss_domain = domain_criterion(domain_out_s, ds) + domain_criterion(domain_out_t, dt)
            loss = loss_label + domain_weight * loss_domain
            loss_domain_val = loss_domain.item()

        loss.backward()
        optimizer.step()

        total_label_loss += loss_label.item()
        total_domain_loss += loss_domain_val

    print(f"epoch {epoch+1}/{n_epochs}  label loss: {total_label_loss/n_batches:.4f}  domain loss: {total_domain_loss/n_batches:.4f}  lambda: {lambd:.3f}")


epoch 1/15  label loss: 0.3287  domain loss: 0.0000  lambda: 0.000
epoch 2/15  label loss: 0.1116  domain loss: 0.0000  lambda: 0.000
epoch 3/15  label loss: 0.0729  domain loss: 0.0000  lambda: 0.000
epoch 4/15  label loss: 0.0519  domain loss: 0.1700  lambda: 0.000
epoch 5/15  label loss: 0.2068  domain loss: 3.7130  lambda: 0.205
epoch 6/15  label loss: 0.0970  domain loss: 1.0869  lambda: 0.394
epoch 7/15  label loss: 0.4270  domain loss: 4.0646  lambda: 0.555
epoch 8/15  label loss: 0.2254  domain loss: 0.0319  lambda: 0.682
epoch 9/15  label loss: 0.1293  domain loss: 0.3044  lambda: 0.779
epoch 10/15  label loss: 0.1243  domain loss: 0.8382  lambda: 0.848
epoch 11/15  label loss: 0.1410  domain loss: 1.4049  lambda: 0.897
epoch 12/15  label loss: 0.0757  domain loss: 1.3765  lambda: 0.931
epoch 13/15  label loss: 0.0565  domain loss: 1.3787  lambda: 0.954
epoch 14/15  label loss: 0.0691  domain loss: 1.3230  lambda: 0.969
epoch 15/15  label loss: 0.0397  domain loss: 1.3826  lam

In [53]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        label_out, _ = model(xb, lambd=0.0)
        preds = label_out.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

print("DANN trained underhang, tested overhang accuracy:", (all_preds == all_true).mean())


DANN trained underhang, tested overhang accuracy: 0.36225405135143085


In [54]:
print(classification_report(le.inverse_transform(all_true), le.inverse_transform(all_preds)))


              precision    recall  f1-score   support

  ball_fault       1.00      0.29      0.45     16577
  cage_fault       0.40      0.85      0.55     22748
      normal       0.00      0.00      0.00      5929
  outer_race       0.04      0.03      0.03     22748

    accuracy                           0.36     68002
   macro avg       0.36      0.29      0.26     68002
weighted avg       0.39      0.36      0.30     68002



In [55]:
cm_dann = confusion_matrix(le.inverse_transform(all_true), le.inverse_transform(all_preds), labels=labels_order)
pd.DataFrame(cm_dann, index=labels_order, columns=labels_order)


,ball_fault,cage_fault,normal,outer_race
ball_fault,4747,274,2020,9536
cage_fault,0,19302,11,3435
normal,0,5929,0,0
outer_race,0,22159,4,585


In [57]:
results_d["DANN v3 "] = (all_preds == all_true).mean()
pd.Series(results_d).sort_values(ascending=False)


log+CORAL(hand features)                    0.519300
DANN v3 (fresh model, fixed optimizer)      0.362254
DANN v3                                     0.362254
DANN v2 (warmup + wider + down-weighted)    0.332387
raw-signal CNN                              0.298212
DANN (domain-adversarial CNN)               0.234831
dtype: float64